# GLM V3 — the reduced core model (PFC): 2×2 + a goal-progress-only arm

Engine [`glm_analysis_v3.py`](glm_analysis_v3.py) — seeded from `glm_analysis_v2.py`, which is left
untouched so every production fit stays reproducible; `glm_v3_synthetics.py` control 1 asserts that V3
under its defaults reproduces V2 to `max|diff| = 0`. Methods, caveats and results: [`GLM_V3.md`](GLM_V3.md).
LEC mirror: [`LEC_glm_v3.ipynb`](../../code/LEC_glm_v3.ipynb).

**The question.** The production 16-regressor fit carries **nine within-leg regressors that are all
functions of one latent — where the animal is in the current leg** (`goal_progress`,
`goal_progress_distance`, `time_from_reward`, `time_to_reward`, `distance_*`, `time_since_A`,
`time_to_A`, `progress_since_A`). Unique-variance measures (Δr², CPD) attribute shared variance to
*none* of them, so each looks tiny — `goal_progress` is −0.00015 after bias correction in LEC and
`task_state` is last in both datasets. Is that dilution by collinearity, or absence of signal?

**The design.** Five regressor groups — `place`, `goal_progress` (time version), `speed`,
`acceleration`, `time_from_reward` — fitted as a **2×2** over the coding of `time_from_reward` and
the leg cap, plus a **goal-progress-only** sanity arm at each cap:

| | tfr **decile** (10 quantile bins, per-recday edges — the v2 coding) | tfr **uniform** (10 equal-width bins in *seconds* over `[0, cap]`, the same edges everywhere) |
|---|---|---|
| **cap 30 s** | `…_cap30s_tfrD10b` | `…_cap30s_tfrU10b` (3 s bins) — the user's literal design |
| **cap 60 s** | `…_cap60s_tfrD10b` — **same rows and same coding as production**; differs from full-16 only in the regressor set | `…_cap60s_tfrU10b` (6 s bins) |
| gp-only | `core_progress_only__…_cap30s` | `core_progress_only__…_cap60s` |

Everything else is the production engine: linear/Gaussian, leave-one-session-out CV, 250 ms binned
aggregation, mixed reference coding, Freedman–Lane permutation null (100), CPD **and** Δr² stored with
their bias-corrected variants. `speed` and `acceleration` stay decile in every arm, so the tfr coding
is the only thing that differs between the two columns.

The design gives four **one-factor-at-a-time** comparisons: *regressor set* (full-16 → core-5 at
decile/60, same rows and coding), *tfr coding* (decile → uniform at each cap), *cap* (30 → 60 at each
coding), and *adding absolute time* (gp-only → gp+tfr).

**Read before any bar** (the full argument is `GLM_V3.md` §Caveats):

1. **Bigger CPDs here are arithmetic, not evidence.** Dropping collinear regressors reassigns their
   shared variance to the survivors. The informative number is *how much* bigger (the shared-variance
   budget), read off decile/60 against full-16 on identical rows — and the gp/tfr split.
2. **gp vs tfr in this model *is* the time-vs-progress dissociation.** Within a leg of duration D,
   `tfr = gp × D`; the two are identified only through leg-duration variability. The joint
   `progress_or_time` block is the honest "within-leg structure, however coded" number.
3. **With the pokes out, reward consumption lands in tfr's first bin.** A big tfr CPD is partly
   consumption; the β profile (§8) shows whether the first bin dominates.
4. **The two codings fail in opposite ways.** Uniform: late bins are sparse and a long-leg indicator.
   Decile: equally populated, but the edges differ in every recday, so a decile coefficient is not
   comparable across recdays or datasets. Both occupancy tables are in §2.
5. **The Freedman–Lane H0 is "g adds nothing beyond the *other regressors in this design*".** With
   four others instead of fifteen it is a weaker H0, so frac_sig rises partly by construction.
6. **The reduced model is a worse model with bigger CPDs** — compare `r2_cv` (§3) before reading any CPD.
7. **Mice, not recdays, are the unit of inference**; every bar is recday median → mouse mean → mean
   over mice, with the mice shown.
8. **gp-only means "within-leg structure of any kind"**, not phase-locking: a pure fixed-latency cell
   scores as a progress cell there (synthetic control 11).

In [1]:
import os, sys, pickle, subprocess, importlib, time, shlex
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = os.path.abspath('../..')
sys.path.insert(0, os.path.join(REPO, "mFC_data", "code"))
import glm_analysis_v3 as glm, w1_refit as w, run_glm_batch as rb, pfc_glm_plots as P

FIGDIR = os.path.join(REPO, 'mFC_data', 'data', 'figures', 'glm_v3')
os.makedirs(FIGDIR, exist_ok=True)
print('engine', glm.GLM_VERSION)

# 0.1 mirror parity: refuse to read results if code/ and mFC_data/code/ have drifted
par = subprocess.run([sys.executable, 'code/check_mirror_parity.py'], cwd=REPO, capture_output=True, text=True)
print(par.stdout.strip().splitlines()[0] if par.stdout.strip() else par.stderr[-500:])
assert par.returncode == 0, 'mirror parity failed -- fix before reading any result'

# 0.2 synthetic controls (refresh with `python mFC_data/code/glm_v3_synthetics.py`)
syn_path = os.path.join(REPO, "mFC_data", "data", 'processed_data', 'glm_v3_synthetics.pkl')
if os.path.exists(syn_path):
    syn = pickle.load(open(syn_path, 'rb'))
    display(pd.DataFrame([{'control': k, 'pass': v['pass'], 'elapsed_s': v['elapsed_s']}
                          for k, v in syn['results'].items()]))
    assert all(v['pass'] for v in syn['results'].values()), 'a synthetic control failed -- see GLM_V3.md section 8'
else:
    print('no synthetics pickle yet -- run glm_v3_synthetics.py')

engine v3


MIRROR PARITY: OK  (glm_cv, glm_v3_synthetics byte-identical; glm_analysis_v2/v3 shared definitions have identical code; plot block identical; registry test identical)


,control,pass,elapsed_s
0,2 fixed-range edges,True,0.0
1,9 registry / stale list,True,0.0
2,"1 v3 defaults == v2 (matched-13, cap 60)",True,155.5
3,"1b v3 decile arm == v2 (core-5, cap 60)",True,49.5
4,"3 latency cell -> tfr, peak at 6-9 s",True,0.0
5,4 phase cell -> gp,True,0.0
6,5 consumption cell -> first tfr bin,True,0.0
7,6 place cell -> place only,True,0.0
8,7 noise cells: FL null calibration,True,0.0
9,8 bin occupancy == trial-times measurement,True,0.1


## 1. The arms and their cache names

Every axis that changes a fit is in the section name (`w1_refit.section_name` + `arm_extra`), and
`recday_registry.is_post_refit_section` must match it — otherwise the stale-cache list re-arms and
`ly05_20250618_20250619` silently vanishes from the fit at load time (it did, once).

**Cap cost and occupancy, measured from the trial times before any fit** (all 25 recdays, uniform 3 s bins):

| | LEC | PFC |
|---|---|---|
| leg duration median / p90 / p99 (s) | 9.2 / 25.6 / 98.4 | 7.5 / 17.2 / 49.8 |
| rows a 30 s cap keeps, relative to the 60 s cap | 80.2 % | 90.6 % |
| rows with tfr > 30 s under the 60 s cap | 5.1 % | 2.3 % |
| in-range rows in bin 0–3 s / bin 27–30 s (60 s cap) | 26.4 % / 1.7 % | 32.6 % / 0.8 % |
| in-range rows in bin 27–30 s (30 s cap) | **0.2 %** | 0.1 % |
| legs reaching 9 s / 18 s / 27 s | 50 % / 16 % / 7 % | 38 % / 9 % / 3 % |

So with the range tied to the cap, the top 3 s bin at cap 30 holds ~35 rows per LEC recday: its β is
noise that still costs a parameter. §2 reports the occupancy of every arm from the fits themselves.

In [2]:
RUN_MODE = 'load'      # 'load' reads the merged fits; 'slurm' submits 6 arms x 25 recdays = 150 PFC jobs; 'local' fits in-process (slow)
WIDTH, SCHEME, REGSET = 250, 'decile', 'matched'   # `matched` on BOTH datasets: the reduced designs need nothing PFC lacks
PERMUTATIONS, CV_PERMS = 100, 100
CAPS = (30.0, 60.0)
TFR_SCHEMES = ('decile', 'uniform')
CORE5 = list(w.SECTIONS['core_progress_time']['regressors'])
CORE4 = list(w.SECTIONS['core_progress_only']['regressors'])

ARMS = {}   # (section, cap_s, tfr_scheme) -> arm flags
for cap in CAPS:
    for sch in TFR_SCHEMES:
        ARMS[('core_progress_time', cap, sch)] = ['--section', 'core_progress_time', '--leg-cap-s', f'{cap:g}', '--tfr-scheme', sch]
    ARMS[('core_progress_only', cap, None)] = ['--section', 'core_progress_only', '--leg-cap-s', f'{cap:g}']

COMMON = ['--width-ms', str(WIDTH), '--scheme', SCHEME, '--regset', REGSET,
          '--permutations', str(PERMUTATIONS), '--cv-perms', str(CV_PERMS)]

def sect_for(key):
    sec, cap, sch = key
    extra = w.arm_extra(w.SECTIONS[sec], leg_cap_s=cap, tfr_scheme=sch or 'uniform', tfr_bins=10)
    return w.section_name(sec, width_ms=WIDTH, scheme=SCHEME, regset=REGSET, extra=extra)

def label(key):
    sec, cap, sch = key
    return f"gp-only/{cap:g}" if sec == 'core_progress_only' else f"gp+tfr {sch}/{cap:g}"

for k in ARMS:
    print(f'{label(k):22s} {sect_for(k):62s}  {" ".join(ARMS[k])}')
FULL = w.section_name('all_regressors', width_ms=WIDTH, scheme=SCHEME, regset="matched")
print('production comparator:', FULL)

gp+tfr decile/30       core_progress_time__matched_250ms_decile_cap30s_tfrD10b         --section core_progress_time --leg-cap-s 30 --tfr-scheme decile
gp+tfr uniform/30      core_progress_time__matched_250ms_decile_cap30s_tfrU10b         --section core_progress_time --leg-cap-s 30 --tfr-scheme uniform
gp-only/30             core_progress_only__matched_250ms_decile_cap30s                 --section core_progress_only --leg-cap-s 30
gp+tfr decile/60       core_progress_time__matched_250ms_decile_cap60s_tfrD10b         --section core_progress_time --leg-cap-s 60 --tfr-scheme decile
gp+tfr uniform/60      core_progress_time__matched_250ms_decile_cap60s_tfrU10b         --section core_progress_time --leg-cap-s 60 --tfr-scheme uniform
gp-only/60             core_progress_only__matched_250ms_decile_cap60s                 --section core_progress_only --leg-cap-s 60
production comparator: all_regressors__matched_250ms_decile


## 2. Fit

One SLURM job per recday per arm, each writing a shard; an explicit `--merge` per arm combines them and
refuses mismatched config stamps (`engine`, `leg_cap_s`, `tfr_scheme`, `tfr_bins`, `tfr_range_s` are
stamped for the V3 arms). `submit_glm_pfc.sh` runs `check_mirror_parity.py` before submitting anything.

In [3]:
def _n_queued():
    q = subprocess.run(['squeue', '--me', '--noheader'], capture_output=True, text=True).stdout
    return len([l for l in q.split('\n') if 'glm_' in l])

if RUN_MODE == 'slurm':
    for k in ARMS:
        cmd = ['bash', 'sbatch_files/submit_glm_pfc.sh'] + ARMS[k] + COMMON
        print('$', ' '.join(shlex.quote(c) for c in cmd))
        out = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
        print(out.stdout[-400:], out.stderr[-400:])
        assert out.returncode == 0, 'submission refused (parity?)'
    while True:
        n = _n_queued(); print(f'  {n} job(s) still queued/running', flush=True)
        if n == 0: break
        time.sleep(120)
    for k in ARMS:
        subprocess.run([sys.executable, 'mFC_data/code/run_glm_batch.py', '--merge'] + ARMS[k] + COMMON, cwd=REPO, check=True)

elif RUN_MODE == 'local':
    import argparse
    for k in ARMS:
        ap = argparse.ArgumentParser(); rb.add_config_args(ap)
        args = ap.parse_args(ARMS[k] + COMMON)
        for rd in w.pfc_recdays():
            rb.fit_one(rd, args)
        rb.merge(args)

elif RUN_MODE != 'load':
    raise ValueError(f'RUN_MODE must be slurm/local/load, got {RUN_MODE!r}')

In [4]:
res = {k: glm.load_glm_results(rb.DEFAULT_OUT, sect_for(k), apply_exclusions=True, verbose=False) for k in ARMS}
fits = {k: r['cv_results'] for k, r in res.items() if 'cv_results' in r}
missing = [sect_for(k) for k in ARMS if k not in fits]
assert not missing, f'arms not merged yet: {missing}'

rows = []
for k, r in res.items():
    cv = r['cv_results']
    
    rows.append({'arm': label(k), 'section': sect_for(k), 'recdays': len(cv),
                 'engine': ','.join(sorted({x.get('engine', 'v2') for x in cv.values()})),
                 'rows/recday (median)': int(np.median([x['n_rows'] for x in cv.values()])),
                 'n_cols': sorted({x['rank']['n_cols'] for x in cv.values()}),
                 'full rank': all(x['rank']['full_rank'] for x in cv.values()),
                 'neurons': sum(len(x['r2_cv']) for x in cv.values())})
display(pd.DataFrame(rows))
full = glm.load_glm_results(rb.DEFAULT_OUT, FULL, apply_exclusions=True, verbose=False)
cv_full = full['cv_results']
print(f'{FULL}: {len(cv_full)} recdays (production, engine v2), {sum(len(x["r2_cv"]) for x in cv_full.values())} neurons')

,arm,section,recdays,engine,rows/recday (median),n_cols,full rank,neurons
0,gp+tfr decile/30,core_progress_time__matched_250ms_decile_cap30...,25,v3,22378,[57],True,1252
1,gp+tfr uniform/30,core_progress_time__matched_250ms_decile_cap30...,25,v3,22378,[57],False,1252
2,gp-only/30,core_progress_only__matched_250ms_decile_cap30s,25,v3,22378,[48],True,1252
3,gp+tfr decile/60,core_progress_time__matched_250ms_decile_cap60...,25,v3,24554,[57],True,1252
4,gp+tfr uniform/60,core_progress_time__matched_250ms_decile_cap60...,25,v3,24554,[57],False,1252
5,gp-only/60,core_progress_only__matched_250ms_decile_cap60s,25,v3,24554,[48],True,1252


all_regressors__matched_250ms_decile: 25 recdays (production, engine v2), 1252 neurons


### 2.1 Bin occupancy from the fits

For every arm, per bin: the median lower edge in seconds (and its spread across recdays — zero for
uniform, the point of uniform; tens of seconds for decile, the point *against* decile for cross-recday
comparison), the share of design rows, and the share of legs that reach the bin. Rows whose block-mean
tfr fell beyond the fixed range (250 ms windows straddling a reward, contaminated by the previous leg's
tail) are coded as bin 0 and counted. A bin under 0.5 % of rows is flagged: its coefficient is noise.

In [5]:
def occupancy_table(cv, name='time_from_reward'):
    occ = [r['bin_occupancy'][name] for r in cv.values() if r.get('bin_occupancy', {}).get(name)]
    fr = np.array([o['frac_rows'] for o in occ], float)
    legs = np.array([np.asarray(o['legs_reaching'], float) / max(o['n_legs'], 1) for o in occ])
    edges = np.array([o['edges_s'] for o in occ], float)[:, 1:-1]
    df = pd.DataFrame({'bin': np.arange(fr.shape[1]),
                       'lower edge s (median over recdays)': np.r_[0.0, np.median(edges, 0)],
                       'edge spread across recdays (max-min, s)': np.r_[0.0, edges.max(0) - edges.min(0)],
                       'rows %': 100 * fr.mean(0), 'legs reaching %': 100 * legs.mean(0)})
    df['flag <0.5% rows'] = df['rows %'] < 0.5
    n_out = int(sum(o['n_out_of_range'] for o in occ)); n_rows = int(sum(o['n_rows'] for o in occ))
    return df, n_out, n_rows

OCC = {}
for k in ARMS:
    if k[0] != 'core_progress_time':
        continue
    df, n_out, n_rows = occupancy_table(fits[k])
    OCC[k] = (df, n_out, n_rows)
    print(f'== {label(k)}   ({sect_for(k)})   rows beyond the fixed range coded as bin 0: {n_out} of {n_rows} ({100 * n_out / max(n_rows, 1):.2f}%)')
    display(df.round(2))

== gp+tfr decile/30   (core_progress_time__matched_250ms_decile_cap30s_tfrD10b)   rows beyond the fixed range coded as bin 0: 0 of 545508 (0.00%)


,bin,lower edge s (median over recdays),"edge spread across recdays (max-min, s)",rows %,legs reaching %,flag <0.5% rows
0,0,0.00,0.00,10.53,100.00,False
1,1,1.01,0.95,9.83,99.81,False
2,2,1.86,1.85,9.84,99.22,False
3,3,2.71,2.74,9.80,97.55,False
4,4,3.61,3.70,9.85,93.09,False
5,5,4.56,4.70,9.82,84.75,False
6,6,5.61,5.80,9.84,72.92,False
7,7,6.96,7.05,9.83,57.77,False
8,8,8.84,8.58,9.84,41.26,False
9,9,11.71,10.33,10.84,22.87,False


== gp+tfr uniform/30   (core_progress_time__matched_250ms_decile_cap30s_tfrU10b)   rows beyond the fixed range coded as bin 0: 14 of 545508 (0.00%)


,bin,lower edge s (median over recdays),"edge spread across recdays (max-min, s)",rows %,legs reaching %,flag <0.5% rows
0,0,0.0,0.0,33.42,100.00,False
1,1,3.0,0.0,27.83,95.88,False
2,2,6.0,0.0,16.96,67.62,False
3,3,9.0,0.0,9.57,41.28,False
4,4,12.0,0.0,5.41,24.03,False
5,5,15.0,0.0,3.15,14.44,False
6,6,18.0,0.0,1.90,9.01,False
7,7,21.0,0.0,1.08,5.39,False
8,8,24.0,0.0,0.53,2.96,False
9,9,27.0,0.0,0.14,1.20,True


== gp+tfr decile/60   (core_progress_time__matched_250ms_decile_cap60s_tfrD10b)   rows beyond the fixed range coded as bin 0: 0 of 601583 (0.00%)


,bin,lower edge s (median over recdays),"edge spread across recdays (max-min, s)",rows %,legs reaching %,flag <0.5% rows
0,0,0.00,0.00,10.58,100.00,False
1,1,1.06,1.21,9.76,99.81,False
2,2,1.94,2.38,9.88,99.00,False
3,3,2.84,3.60,9.79,96.73,False
4,4,3.79,4.90,9.83,90.91,False
5,5,4.84,6.30,9.82,80.93,False
6,6,6.21,8.00,9.82,67.28,False
7,7,7.71,10.12,9.85,51.70,False
8,8,9.64,12.70,9.83,34.09,False
9,9,13.52,16.25,10.83,16.34,False


== gp+tfr uniform/60   (core_progress_time__matched_250ms_decile_cap60s_tfrU10b)   rows beyond the fixed range coded as bin 0: 3 of 601583 (0.00%)


,bin,lower edge s (median over recdays),"edge spread across recdays (max-min, s)",rows %,legs reaching %,flag <0.5% rows
0,0,0.0,0.0,57.30,100.00,False
1,1,6.0,0.0,25.15,68.28,False
2,2,12.0,0.0,8.89,25.98,False
3,3,18.0,0.0,4.03,11.63,False
4,4,24.0,0.0,2.08,5.95,False
5,5,30.0,0.0,1.19,3.20,False
6,6,36.0,0.0,0.74,1.90,False
7,7,42.0,0.0,0.41,1.09,True
8,8,48.0,0.0,0.16,0.60,True
9,9,54.0,0.0,0.03,0.15,True


## 3. Does the model explain the data at all — and how much worse is the reduced one?

`r2_cv` per neuron (held-out, leave-one-session-out). The reduced designs drop regressors that carried
unique variance (in LEC, `head_direction` was 2nd and `poke_unrewarded` 4th after bias correction) — but
they also drop the held-out parameter penalty of ~100 columns that were largely fitting noise. Which
effect wins is an empirical question, and this panel answers it; do not assume the direction. Whatever
the CPDs below say, this is the panel that keeps "bigger CPD" from being read as "better model" — and
"slightly better `r2_cv`" from being read as "the dropped regressors carried nothing".

In [6]:
fig = P.plot_model_fit(cv_full, out_path=f'{FIGDIR}/pfc_model_fit_full16.pdf'); fig.suptitle('production ' + FULL, fontsize=7); plt.show()
for k in ARMS:
    fig = P.plot_model_fit(fits[k], out_path=f"{FIGDIR}/pfc_model_fit_{sect_for(k).split('__')[1]}.pdf")
    fig.suptitle(label(k), fontsize=7); plt.show()
r2tab = pd.DataFrame({('production full', ''): P._r2_per_mouse(cv_full).set_index('mouse')['value'],
                      **{(label(k), ''): P._r2_per_mouse(fits[k]).set_index('mouse')['value'] for k in ARMS}})
r2tab.loc['mean over mice'] = r2tab.mean()
print('median r2_cv per recday -> mean per mouse:'); display(r2tab.round(4))

median r2_cv per recday -> mean per mouse:


,production full,gp+tfr decile/30,gp+tfr uniform/30,gp-only/30,gp+tfr decile/60,gp+tfr uniform/60,gp-only/60
,,,,,,,
mouse,,,,,,,
ab03,0.0632,0.0719,0.0692,0.0632,0.0696,0.0630,0.0630
ah03,0.0099,0.0141,0.0134,0.0146,0.0184,0.0182,0.0157
ah04,0.0103,0.0158,0.0146,0.0135,0.0160,0.0129,0.0136
ah07,0.0383,0.0462,0.0444,0.0398,0.0459,0.0402,0.0386
me08,0.0565,0.0713,0.0675,0.0649,0.0689,0.0674,0.0634
me10,0.0193,0.0406,0.0399,0.0407,0.0408,0.0387,0.0392
me11,0.0904,0.1020,0.0989,0.0974,0.0960,0.0906,0.0911
mean over mice,0.0411,0.0517,0.0497,0.0477,0.0508,0.0473,0.0464


## 4. Ranking per arm — raw CPD (primary), Δr², and both bias-corrected

**How a bar is computed** (same chain as the production notebooks, `glm_plots.per_mouse_stat`):
per neuron, RSS and TSS are summed over the held-out folds and the ratio formed once — CPD =
ΔRSS/RSS_reduced, Δr² = ΔRSS/TSS; per recday the **median** over neurons; per mouse the **mean** over
its recdays; the bar is the **mean over mice** and the dots are the mice. Recdays of one mouse are the
same probe re-sorted, so they are repeated measures and are never pooled as neurons.

**Bias correction.** Held-out CPD and Δr² carry a downward penalty ∝ the regressor's column count
(the k extra parameters of the full model fit training noise). The Freedman–Lane null measures that
penalty per neuron, so `corrected = observed − null_mean`. In the reduced designs every block has 9
columns except `place` (20), so the correction moves less than it did in the 16-regressor fit — but
zero is still not the reference, the null centre is.

In [7]:
for k in ARMS:
    cv = fits[k]; short = sect_for(k).split('__')[1]
    for val in P.VALUE_OPTIONS:
        fig = P.plot_regressor_ranking(cv, value=val, out_path=f'{FIGDIR}/pfc_{short}_ranking_{P._VALUE_SHORT[val]}.pdf')
        fig.suptitle(label(k), fontsize=7)
        if val in ('cpd_cv', 'cpd_corrected'):
            plt.show()
        else:
            plt.close(fig)
    print(f'--- {label(k)}: raw CPD, frac_sig = Freedman-Lane on CPD')
    display(P.summary_table(cv, value='cpd_cv', p_stat='cpd').round(5))

--- gp+tfr decile/30: raw CPD, frac_sig = Freedman-Lane on CPD


,CPD,CPD_sd,frac_sig,n_mice,family
regressor,,,,,
place,0.00736,0.00843,0.67841,7,space
goal_progress,0.00201,0.00182,0.65414,7,task
time_from_reward,0.00195,0.00220,0.64420,7,time
acceleration,0.00223,0.00255,0.65256,7,motor
speed,0.00663,0.00752,0.70027,7,motor


--- gp+tfr uniform/30: raw CPD, frac_sig = Freedman-Lane on CPD


,CPD,CPD_sd,frac_sig,n_mice,family
regressor,,,,,
place,0.00733,0.00847,0.67403,7,space
goal_progress,0.00412,0.00310,0.74698,7,task
time_from_reward,0.00059,0.00154,0.51104,7,time
acceleration,0.00224,0.00261,0.65142,7,motor
speed,0.00670,0.00749,0.69823,7,motor


--- gp-only/30: raw CPD, frac_sig = Freedman-Lane on CPD


...

,CPD,CPD_sd,frac_sig,n_mice,family
regressor,,,,,
place,0.00801,0.00897,0.69423,7,space
goal_progress,0.00155,0.00158,0.64568,7,task
time_from_reward,0.00289,0.00226,0.66287,7,time
acceleration,0.00233,0.00259,0.67809,7,motor
speed,0.00654,0.00695,0.70853,7,motor


--- gp+tfr uniform/60: raw CPD, frac_sig = Freedman-Lane on CPD


,CPD,CPD_sd,frac_sig,n_mice,family
regressor,,,,,
place,0.00779,0.00877,0.72229,7,space
goal_progress,0.00645,0.00338,0.78715,7,task
time_from_reward,-0.00061,0.00058,0.34477,7,time
acceleration,0.00235,0.00258,0.67678,7,motor
speed,0.00656,0.00710,0.71367,7,motor


--- gp-only/60: raw CPD, frac_sig = Freedman-Lane on CPD


,CPD,CPD_sd,frac_sig,n_mice,family
regressor,,,,,
place,0.00775,0.00905,0.67716,7,space
goal_progress,0.00779,0.00422,0.79706,7,task
acceleration,0.00241,0.00267,0.68601,7,motor
speed,0.00681,0.00709,0.70684,7,motor


## 5. Significant fraction — Freedman–Lane, on CPD and on Δr²

The null: for regressor *g*, fit the reduced model (everything but *g*), circularly shift its residuals
within session, rebuild `y* = ŷ_red + e*`, rerun the whole CV, 100 times; `p = (1 + #{null ≥ obs}) /
(1 + n)`. It tests "*g* adds nothing beyond the other regressors **in this design**". With four others
rather than fifteen that is a weaker H0, so the fractions below are expected to exceed the full-16
fractions for the same neurons — the comparison table says by how much. One-sided: a regressor that
*hurts* held-out prediction is never "significant".

In [8]:
for k in ARMS:
    short = sect_for(k).split('__')[1]
    fig = P.plot_significant_fraction(fits[k], p_stat='cpd', out_path=f'{FIGDIR}/pfc_{short}_frac_sig_cpd.pdf'); fig.suptitle(label(k), fontsize=7); plt.show()
    fig = P.plot_significant_fraction(fits[k], p_stat='delta_r2', out_path=f'{FIGDIR}/pfc_{short}_frac_sig_delta_r2.pdf'); plt.close(fig)

key_d60 = ('core_progress_time', 60.0, 'decile')
tab = {}
for lab, cv in [('full-16 production', cv_full), ('core-5 decile/60', fits[key_d60])]:
    for ps in ('cpd', 'delta_r2'):
        st = P.summary_table(cv, value='cpd_cv', p_stat=ps)
        tab[(lab, f'frac_sig {ps}')] = st.loc[[g for g in CORE5 if g in st.index], 'frac_sig']
print('same rows, same coding, different regressor set -- the frac_sig shift is the H0 changing:')
display(pd.DataFrame(tab).round(3))

/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/code/pfc_glm_plots.py:311: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, a = (plt.subplots(figsize=(4.2, 3.2)) if ax is None else (ax.figure, ax))


/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/code/pfc_glm_plots.py:311: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, a = (plt.subplots(figsize=(4.2, 3.2)) if ax is None else (ax.figure, ax))


same rows, same coding, different regressor set -- the frac_sig shift is the H0 changing:


full-16 production                   core-5 decile/60  \
                       frac_sig cpd frac_sig delta_r2     frac_sig cpd   
regressor                                                                
place                         0.703             0.703            0.694   
goal_progress                 0.473             0.476            0.646   
speed                         0.722             0.722            0.709   
acceleration                  0.632             0.632            0.678   
time_from_reward              0.625             0.625            0.663   

                                    
                 frac_sig delta_r2  
regressor                           
place                        0.694  
goal_progress                0.646  
speed                        0.709  
acceleration                 0.678  
time_from_reward             0.663

## 6. The regressor-set effect: full-16 → core-5, on identical rows and coding

decile/60 uses exactly the production rows (60 s cap, majority-valid windows) and the production tfr
coding, so the only difference from the full fit is the eleven regressors that were dropped. The
shift in each survivor's CPD is the shared variance those eleven were absorbing. Two per-neuron
checks follow, and neither is a guarantee: (i) reduced `r2_cv` ≤ full — *not* implied under
cross-validation, because a 160-column model pays a held-out parameter penalty a 57-column one does
not, so a reduced design can generalise better while explaining less in-sample; (ii) reduced CPD ≥
full for each survivor — expected *on average* (shared variance is reassigned to the survivors) but
noisy per neuron. Read them as descriptions of what happened, not as pass/fail.

In [9]:
# Same rows = identical held-out TSS per neuron (TSS depends only on the rows and the within-session
# centring, not on X). A recday whose production fit was made on different data -- the known case is
# ah10_20250618_20250619, whose session 5 was salvaged on 2026-09-06 after the production fit -- is
# excluded from this comparison and named, rather than silently pooled.
cand = sorted(set(fits[key_d60]) & set(cv_full))
shared = [rd for rd in cand if np.allclose(fits[key_d60][rd]['tss'], cv_full[rd]['tss'])]
excluded = sorted(set(cand) - set(shared))
print(f'{len(shared)} of {len(cand)} recdays share rows with the production fit (TSS identical per neuron)')
for rd in excluded:
    print(f'  excluded {rd}: production n_folds {cv_full[rd]["n_folds"]} vs V3 {fits[key_d60][rd]["n_folds"]} -- the production fit predates a data change for this recday')

for val in ('cpd_cv', 'delta_r2_cv', 'cpd_corrected', 'delta_r2_corrected'):
    fig = P.plot_paired_fits({'full-16': {rd: cv_full[rd] for rd in shared}, 'core-5 decile/60': {rd: fits[key_d60][rd] for rd in shared}},
                             regressors=CORE5, value=val, out_path=f'{FIGDIR}/pfc_full16_vs_core5_{P._VALUE_SHORT[val]}.pdf')
    plt.show()

rows = []
for rd in shared:
    a, b = cv_full[rd], fits[key_d60][rd]
    row = {'recday': rd, 'frac neurons r2 reduced <= full': float(np.mean(b['r2_cv'] <= a['r2_cv'] + 1e-12))}
    for g in CORE5:
        row[f'frac CPD reduced >= full: {g}'] = float(np.mean(np.asarray(b['cpd_cv'][g]) >= np.asarray(a['cpd_cv'][g]) - 1e-12))
    rows.append(row)
san = pd.DataFrame(rows).set_index('recday')
print('arithmetic sanity (should be near 1 everywhere):'); display(san.agg(['mean', 'min']).round(3))

def per_mouse_group(cv, g, value='cpd_cv'):
    df = pd.DataFrame([{'mouse': rd.split('_')[0], 'v': float(np.nanmedian(P.resolve_value(r, value)[g]))} for rd, r in cv.items() if g in P.resolve_value(r, value)])
    return df.groupby('mouse')['v'].mean()
print('joint block progress_or_time (gp and tfr dropped together), CPD per mouse:')
display(pd.DataFrame({label(k): per_mouse_group(fits[k], 'progress_or_time') for k in ARMS if k[0] == 'core_progress_time'}).round(4))

25 of 25 recdays share rows with the production fit (TSS identical per neuron)


arithmetic sanity (should be near 1 everywhere):


,frac neurons r2 reduced <= full,frac CPD reduced >= full: place,frac CPD reduced >= full: goal_progress,frac CPD reduced >= full: speed,frac CPD reduced >= full: acceleration,frac CPD reduced >= full: time_from_reward
mean,0.203,0.415,0.672,0.581,0.675,0.653
min,0.000,0.167,0.000,0.000,0.000,0.346


joint block progress_or_time (gp and tfr dropped together), CPD per mouse:


,gp+tfr decile/30,gp+tfr uniform/30,gp+tfr decile/60,gp+tfr uniform/60
mouse,,,,
ab03,0.0220,0.0198,0.0204,0.0146
ah03,0.0072,0.0058,0.0080,0.0056
ah04,0.0057,0.0045,0.0054,0.0025
ah07,0.0138,0.0125,0.0138,0.0099
me08,0.0158,0.0132,0.0144,0.0103
me10,0.0042,0.0039,0.0039,0.0024
me11,0.0127,0.0111,0.0118,0.0083


## 7. Coding and cap effects — the 2×2

One figure per regressor: rows are caps, the two codings side by side; points are mice joined across
codings. The coding effect is read along a row, the cap effect down a column. Two reminders: the 30 s
arms sit on ~80 % of the 60 s rows (LEC; 91 % PFC), so a cap effect is partly a row-set effect; and
the top uniform bins at cap 30 are nearly empty (§2.1), so a uniform-vs-decile difference in
`time_from_reward` is partly the parameter penalty of those bins.

In [10]:
arms2x2 = {(cap, sch): fits[('core_progress_time', cap, sch)] for cap in CAPS for sch in TFR_SCHEMES}
for g in CORE5:
    for val in ('cpd_cv', 'delta_r2_cv'):
        fig = P.plot_factorial_grid(arms2x2, g, value=val, out_path=f'{FIGDIR}/pfc_grid_{g}_{P._VALUE_SHORT[val]}.pdf')
        if val == 'cpd_cv': plt.show()
        else: plt.close(fig)
# paired: coding effect at each cap, cap effect at each coding
for cap in CAPS:
    fig = P.plot_paired_fits({f'decile/{cap:g}': arms2x2[(cap, 'decile')], f'uniform/{cap:g}': arms2x2[(cap, 'uniform')]}, regressors=CORE5, value='cpd_cv',
                             out_path=f'{FIGDIR}/pfc_coding_effect_cap{cap:g}.pdf'); plt.show()
for sch in TFR_SCHEMES:
    fig = P.plot_paired_fits({f'{sch}/30': arms2x2[(30.0, sch)], f'{sch}/60': arms2x2[(60.0, sch)]}, regressors=CORE5, value='cpd_cv',
                             out_path=f'{FIGDIR}/pfc_cap_effect_{sch}.pdf'); plt.show()

## 7b. The goal-progress-only sanity arm, and the gp decomposition

Goal-progress tuning is visible by eye in the data, so a design in which `goal_progress` is the
**only** within-leg regressor should show it plainly. It does not say the structure is phase-locked:
with no other within-leg regressor, gp absorbs elapsed time, distance, consumption and leg-type effects
alike — synthetic control 11 shows a pure fixed-latency cell scoring as a progress cell here. So read the
three gp CPDs at a cap together: **gp-only → gp+tfr(decile) → gp+tfr(uniform)**. The drop from the
first to the others is the variance gp shares with absolute time; the `r2_cv` panel shows what adding
tfr buys. Arithmetic sanity: adding a competitor cannot raise a regressor's *unique* variance except by
noise, so gp CPD in gp-only ≥ gp CPD in gp+tfr per neuron in the large majority.

In [11]:
for cap in CAPS:
    kgp = ('core_progress_only', cap, None)
    print(f'--- gp-only, cap {cap:g} s')
    display(P.summary_table(fits[kgp], value='cpd_cv', p_stat='cpd').round(5))
    trio = {f'gp-only/{cap:g}': fits[kgp], f'gp+tfr decile/{cap:g}': arms2x2[(cap, 'decile')], f'gp+tfr uniform/{cap:g}': arms2x2[(cap, 'uniform')]}
    for val in ('cpd_cv', 'delta_r2_cv'):
        fig = P.plot_paired_fits(trio, regressors=['goal_progress', 'place'], value=val, out_path=f'{FIGDIR}/pfc_gp_decomposition_cap{cap:g}_{P._VALUE_SHORT[val]}.pdf'); plt.show()
    frac = []
    for rd in fits[kgp]:
        if rd not in arms2x2[(cap, 'uniform')]: continue
        a = np.asarray(fits[kgp][rd]['cpd_cv']['goal_progress']); b = np.asarray(arms2x2[(cap, 'uniform')][rd]['cpd_cv']['goal_progress'])
        frac.append(float(np.mean(a >= b - 1e-12)))
    print(f'  frac neurons with gp CPD(gp-only) >= gp CPD(gp+tfr uniform): mean {np.mean(frac):.3f}, min {np.min(frac):.3f} over recdays')

--- gp-only, cap 30 s


,CPD,CPD_sd,frac_sig,n_mice,family
regressor,,,,,
place,0.00719,0.00826,0.66769,7,space
goal_progress,0.00872,0.00439,0.82755,7,task
acceleration,0.00229,0.00265,0.65498,7,motor
speed,0.00688,0.00774,0.70292,7,motor


  frac neurons with gp CPD(gp-only) >= gp CPD(gp+tfr uniform): mean 0.729, min 0.000 over recdays
--- gp-only, cap 60 s


,CPD,CPD_sd,frac_sig,n_mice,family
regressor,,,,,
place,0.00775,0.00905,0.67716,7,space
goal_progress,0.00779,0.00422,0.79706,7,task
acceleration,0.00241,0.00267,0.68601,7,motor
speed,0.00681,0.00709,0.70684,7,motor


  frac neurons with gp CPD(gp-only) >= gp CPD(gp+tfr uniform): mean 0.627, min 0.333 over recdays


## 8. Where along time-from-reward do tfr-tuned neurons load?

Reference-coded β per tfr bin (bin 0 = the reference bin, β = 0 by construction) for every neuron
whose tfr CPD beats its Freedman–Lane null, scaled to unit max|β| and averaged — a shape average —
with the bin occupancy underneath. A first-bin peak with the pokes out of the model is consumption
(caveat 3); a late-bin peak sits on the sparsest bins (caveat 4). Under decile coding the x positions
are the median edge over recdays.

In [12]:
col_idx = glm._resolve_regressor_groups(CORE5, parameterization='reference_coded')[0]['time_from_reward']
region_of = None
for (cap, sch), cv in arms2x2.items():
    k = ('core_progress_time', cap, sch)
    fig = P.plot_tfr_beta_profile(res[k]['glm_results'], cv, col_idx, region_of=region_of,
                                  
                                  out_path=f"{FIGDIR}/pfc_tfr_beta_profile_{sect_for(k).split('__')[1]}.pdf")
    fig.suptitle(label(k), fontsize=7); plt.show()

### 8.1 The same figure for goal progress

Reference-coded β over the 10 equal-width phase bins, per neuron divided by that neuron's own max|β|,
then averaged (± s.e.m.) over the gp-significant neurons of a region.

**How to read bin 0.** It is the first tenth of the leg — the reward-consumption period — and it is the
reference bin: it has no design column, its level sits in the intercept, and each β_k is the difference
in expected firing between bin k and bin 0 with everything else held fixed. So every neuron is at exactly
0 there *by construction* (the band vanishes), a curve above zero means "fires more in that phase than
during consumption", and only the shape across bins 1–9 is data; the offset is not. `center='mean'`
(the `*_meancentred.pdf` variants) subtracts each neuron's mean over the ten bins first so the baseline
is the neuron's own phase-average and bin 0 is not privileged — identical shapes, different baseline.

Goal progress has ~10 % of rows in every bin by construction (measured 9.5–10.4 %), so the lower panel
shows instead where those neurons fire most: the bin of highest β per neuron, **bin 0 included** — a
neuron whose every β is negative fires most during consumption and is counted in bin 0. All six arms:
with `time_from_reward` in the model (four codings/caps) the profile is what phase explains *beyond*
absolute time; in the gp-only arms it is within-leg structure of any kind (caveat 9).

In [13]:
for k in ARMS:
    regs = CORE5 if k[0] == 'core_progress_time' else CORE4
    gp_idx = glm._resolve_regressor_groups(regs, parameterization='reference_coded')[0]['goal_progress']
    fig = P.plot_beta_profile(res[k]['glm_results'], fits[k], gp_idx, 'goal_progress', region_of=region_of,
                              
                              out_path=f"{FIGDIR}/pfc_gp_beta_profile_{sect_for(k).split('__')[1]}.pdf")
    fig.suptitle(label(k), fontsize=7); plt.show()

## 9. LEC regions + PFC, same design

The reduced designs need nothing PFC lacks, so for the first time a regional LEC panel can carry a PFC
column without a matched/full asymmetry: both arms are `--regset matched` and the section names
coincide. The cell below refuses to compare arms whose regressor groups differ. The PFC column is a
whole dataset next to LEC sub-regions, and PFC's leg durations are shorter (median 7.5 vs 9.2 s), so
under uniform coding the same bin holds a different share of rows in the two datasets — the occupancy
tables of both notebooks are the caveat.

In [14]:
sys.path.insert(0, os.path.join(REPO, 'code'))
import glm_plots as LP, anatomy_split as asp
LEC_OUT = os.path.join(REPO, 'data', 'glm_outputs', 'LEC')
unit_regions = asp.load_unit_regions()
lec = {}
for k in ARMS:
    r = glm.load_glm_results(LEC_OUT, sect_for(k), apply_exclusions=True, verbose=False)
    if 'cv_results' in r:
        lec[k] = r['cv_results']
        regs_l = {g for g in next(iter(lec[k].values()))['delta_r2_cv'] if not g.startswith('__')}
        regs_p = {g for g in next(iter(fits[k].values()))['delta_r2_cv'] if not g.startswith('__')}
        assert regs_l == regs_p, f'{sect_for(k)}: designs differ -- LEC only {regs_l - regs_p}, PFC only {regs_p - regs_l}'
print(f'{len(lec)} arms present in both datasets with identical regressor groups')
for k in lec:
    for val in ('cpd_cv', 'delta_r2_cv'):
        fig = LP.plot_regressor_ranking_by_region(lec[k], unit_regions, value=val, extra={'PFC': fits[k]},
                                                  out_path=f"{FIGDIR}/pfc_vs_lec_regions_{sect_for(k).split('__')[1]}_{P._VALUE_SHORT[val]}.pdf")
        fig.suptitle(label(k), fontsize=7); plt.show()
    fig = LP.plot_significant_fraction_by_region(lec[k], unit_regions, p_stat='cpd', extra={'PFC': fits[k]},
                                                 out_path=f"{FIGDIR}/pfc_vs_lec_regions_{sect_for(k).split('__')[1]}_frac_sig.pdf")
    fig.suptitle(label(k), fontsize=7); plt.show()

# pooled LEC (all regions) vs PFC, per mouse, per arm
rows = []
for k in lec:
    for g in P.regressor_names(fits[k]):
        a = P.per_mouse_stat(P.to_long(lec[k], 'cpd_cv'), 'value', 'median'); a = a[a.regressor == g]['value']
        b = P.per_mouse_stat(P.to_long(fits[k], 'cpd_cv'), 'value', 'median'); b = b[b.regressor == g]['value']
        rows.append({'arm': label(k), 'regressor': g, 'LEC': a.mean(), 'LEC n_mice': len(a), 'PFC': b.mean(), 'PFC n_mice': len(b), 'LEC - PFC': a.mean() - b.mean()})
display(pd.DataFrame(rows).set_index(['arm', 'regressor']).round(5))

6 arms present in both datasets with identical regressor groups


LEC  LEC n_mice      PFC  PFC n_mice  \
arm               regressor                                                    
gp+tfr decile/30  place             0.01526           5  0.00736           7   
                  goal_progress    -0.00043           5  0.00201           7   
                  time_from_reward  0.00057           5  0.00195           7   
                  acceleration     -0.00004           5  0.00223           7   
                  speed             0.00182           5  0.00663           7   
gp+tfr uniform/30 place             0.01528           5  0.00733           7   
                  goal_progress     0.00047           5  0.00412           7   
                  time_from_reward -0.00039           5  0.00059           7   
                  acceleration     -0.00002           5  0.00224           7   
                  speed             0.00185           5  0.00670           7   
gp-only/30        place             0.01560           5  0.00719           7   
                  goal_progress     0.00171           5  0.00872           7   
                  acceleration      0.00003           5  0.00229           7   
                  speed             0.00206           5  0.00688           7   
gp+tfr decile/60  place             0.01521           5  0.00801           7   
                  goal_progress    -0.00032           5  0.00155           7   
                  time_from_reward  0.00070           5  0.00289           7   
                  acceleration      0.00017           5  0.00233           7   
                  speed             0.00224           5  0.00654           7   
gp+tfr uniform/60 place             0.01590           5  0.00779           7   
                  goal_progress     0.00093           5  0.00645           7   
                  time_from_reward -0.00102           5 -0.00061           7   
                  acceleration      0.00023           5  0.00235           7   
                  speed             0.00250           5  0.00656           7   
gp-only/60        place             0.01594           5  0.00775           7   
                  goal_progress     0.00128           5  0.00779           7   
                  acceleration      0.00023           5  0.00241           7   
                  speed             0.00251           5  0.00681           7   

                                    LEC - PFC  
arm               regressor                    
gp+tfr decile/30  place               0.00790  
                  goal_progress      -0.00244  
                  time_from_reward   -0.00137  
                  acceleration       -0.00226  
                  speed              -0.00481  
gp+tfr uniform/30 place               0.00795  
                  goal_progress      -0.00365  
                  time_from_reward   -0.00098  
                  acceleration       -0.00226  
                  speed              -0.00485  
gp-only/30        place               0.00841  
                  goal_progress      -0.00701  
                  acceleration       -0.00226  
                  speed              -0.00481  
gp+tfr decile/60  place               0.00720  
                  goal_progress      -0.00187  
                  time_from_reward   -0.00219  
                  acceleration       -0.00216  
                  speed              -0.00430  
gp+tfr uniform/60 place               0.00811  
                  goal_progress      -0.00552  
                  time_from_reward   -0.00041  
                  acceleration       -0.00212  
                  speed              -0.00407  
gp-only/60        place               0.00819  
                  goal_progress      -0.00651  
                  acceleration       -0.00218  
                  speed              -0.00430

## 10. Records

Results go into `mFC_data/code/GLM_V3.md` (this dataset's run) and `code/GLM_V3.md` §10 (the
cross-dataset panel). Nothing here is committed. What this does not license: `GLM_V3.md` §11.